In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap


Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)

In [ ]:

rna = pd.read_csv("../data/processed/rna_pam50.csv").set_index("patient")
meth = pd.read_csv("../data/processed/meth_pam50_knn_imputed.csv", index_col=0)
labels = pd.read_csv("../data/processed/labels_luminal_brca.csv").set_index("patient")
cpg_gene = pd.read_csv("../data/processed/cpg_gene_map.csv")

patients = rna.index.intersection(meth.index).intersection(labels.index)
rna = rna.loc[patients]
meth = meth.loc[patients]
labels = labels.loc[patients]
subtype = labels["subtype"]

print("Patients:", len(patients))
print(subtype.value_counts())

In [ ]:

# RNA violin plots for key PAM50 genes by subtype
KEY_GENES = ["ESR1", "PGR", "MKI67", "BIRC5", "ERBB2", "GRB7", "KRT5", "KRT14"]
available = [g for g in KEY_GENES if g in rna.columns]

COLORS = {"LumA": "#2a9d8f", "LumB": "#e76f51"}

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for ax, gene in zip(axes, available):
    luma = rna.loc[subtype == "LumA", gene].dropna()
    lumb = rna.loc[subtype == "LumB", gene].dropna()

    parts = ax.violinplot([luma, lumb], positions=[0, 1], showmedians=True, showextrema=False)
    for pc, color in zip(parts["bodies"], [COLORS["LumA"], COLORS["LumB"]]):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    parts["cmedians"].set_color("black")
    parts["cmedians"].set_linewidth(1.5)

    rng = np.random.default_rng(42)
    for pos, vals, color in zip([0, 1], [luma, lumb], [COLORS["LumA"], COLORS["LumB"]]):
        idx = rng.choice(len(vals), size=min(80, len(vals)), replace=False)
        jitter = rng.uniform(-0.07, 0.07, size=len(idx))
        ax.scatter(pos + jitter, vals.iloc[idx], s=5, alpha=0.4, color=color)

    _, pval = stats.mannwhitneyu(luma, lumb, alternative="two-sided")
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"

    ax.set_title(f"{gene}  {sig}", fontsize=11)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["LumA", "LumB"])
    ax.set_ylabel("log2 expr.")
    ax.spines[["top", "right"]].set_visible(False)

for ax in axes[len(available):]:
    ax.set_visible(False)

fig.suptitle("PAM50 Gene Expression by Luminal Subtype\n(Mann-Whitney U: * p<0.05, ** p<0.01, *** p<0.001)", fontsize=12)
plt.tight_layout()
plt.savefig("../results/figures/rna_violin_by_subtype.png", dpi=300)
plt.show()

In [ ]:

# PCA of RNA features
scaler = StandardScaler()
rna_scaled = scaler.fit_transform(rna)
pca_rna = PCA(n_components=5, random_state=42)
rna_pcs = pca_rna.fit_transform(rna_scaled)
var_rna = pca_rna.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (px, py) in zip(axes, [(0, 1), (1, 2)]):
    for st in ["LumA", "LumB"]:
        mask = (subtype == st).values
        ax.scatter(rna_pcs[mask, px], rna_pcs[mask, py],
                   c=COLORS[st], label=st, s=18, alpha=0.6, edgecolors="none")
    ax.set_xlabel(f"PC{px+1} ({var_rna[px]:.1f}%)")
    ax.set_ylabel(f"PC{py+1} ({var_rna[py]:.1f}%)")
    ax.set_title(f"RNA PCA: PC{px+1} vs PC{py+1}")
    ax.legend(title="Subtype")
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("PCA of PAM50 RNA Expression (50 genes)", fontsize=12)
plt.tight_layout()
plt.savefig("../results/figures/rna_pca_by_subtype.png", dpi=300)
plt.show()

pd.DataFrame(
    pca_rna.components_[:3].T,
    index=rna.columns,
    columns=["PC1", "PC2", "PC3"]
).assign(PC1_abs=lambda d: d["PC1"].abs()).sort_values("PC1_abs", ascending=False).drop(columns="PC1_abs").to_csv(
    "../results/tables/rna_pca_loadings.csv"
)

In [ ]:

# PCA of methylation features (post-imputation)
meth_clean = meth.dropna(axis=1, how="all")

scaler_m = StandardScaler()
meth_scaled = scaler_m.fit_transform(meth_clean)
pca_meth = PCA(n_components=5, random_state=42)
meth_pcs = pca_meth.fit_transform(meth_scaled)
var_meth = pca_meth.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (px, py) in zip(axes, [(0, 1), (1, 2)]):
    for st in ["LumA", "LumB"]:
        mask = (subtype == st).values
        ax.scatter(meth_pcs[mask, px], meth_pcs[mask, py],
                   c=COLORS[st], label=st, s=18, alpha=0.6, edgecolors="none")
    ax.set_xlabel(f"PC{px+1} ({var_meth[px]:.1f}%)")
    ax.set_ylabel(f"PC{py+1} ({var_meth[py]:.1f}%)")
    ax.set_title(f"Methylation PCA: PC{px+1} vs PC{py+1}")
    ax.legend(title="Subtype")
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("PCA of PAM50 Promoter Methylation (post-imputation)", fontsize=12)
plt.tight_layout()
plt.savefig("../results/figures/meth_pca_by_subtype.png", dpi=300)
plt.show()

In [ ]:

# Meth vs RNA scatter for a selection of key genes (multi-gene panel)
# uses cpg_gene_map to get per-gene mean promoter beta
SCATTER_GENES = ["ESR1", "PGR", "ERBB2", "FOXA1", "MKI67", "KRT5", "BCL2", "MLPH"]
available_scatter = [g for g in SCATTER_GENES if g in rna.columns]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
n_plotted = 0

for ax, gene in zip(axes, available_scatter):
    cpg_ids = cpg_gene.loc[cpg_gene["gene"] == gene, "cpg"].tolist()
    cpg_ids = [c for c in cpg_ids if c in meth_clean.columns]
    if not cpg_ids:
        ax.set_visible(False)
        continue

    mean_beta = meth_clean[cpg_ids].mean(axis=1)
    both = pd.DataFrame({"beta": mean_beta, "rna": rna[gene]}).dropna()
    rho, pval = stats.spearmanr(both["beta"], both["rna"])

    for st in ["LumA", "LumB"]:
        mask = (subtype.loc[both.index] == st).values
        ax.scatter(both.loc[mask, "beta"], both.loc[mask, "rna"],
                   c=COLORS[st], s=8, alpha=0.45, label=st, edgecolors="none")

    ax.set_xlabel("Mean promoter beta")
    ax.set_ylabel("log2 expr.")
    ax.set_title(f"{gene}  rho={rho:.2f}")
    ax.spines[["top", "right"]].set_visible(False)
    n_plotted += 1

for ax in axes[n_plotted:]:
    ax.set_visible(False)

handles = [plt.scatter([], [], c=COLORS[s], label=s) for s in ["LumA", "LumB"]]
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Promoter Methylation vs Gene Expression (Spearman rho)\nnegative rho = epigenetic silencing", fontsize=12)
plt.tight_layout()
plt.savefig("../results/figures/meth_vs_rna_scatter_panel.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:

# UMAP of RNA features
reducer_rna = umap.UMAP(n_components=2, random_state=42)
rna_umap = reducer_rna.fit_transform(rna_scaled)

fig, ax = plt.subplots(figsize=(7, 5))
for st in ["LumA", "LumB"]:
    mask = (subtype == st).values
    ax.scatter(rna_umap[mask, 0], rna_umap[mask, 1],
               c=COLORS[st], label=st, s=18, alpha=0.6, edgecolors="none")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("UMAP of PAM50 RNA Expression (50 genes)")
ax.legend(title="Subtype")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../results/figures/rna_umap_by_subtype.png", dpi=300)
plt.show()

In [ ]:

# UMAP of methylation features (post-imputation)
reducer_meth = umap.UMAP(n_components=2, random_state=42)
meth_umap = reducer_meth.fit_transform(meth_scaled)

fig, ax = plt.subplots(figsize=(7, 5))
for st in ["LumA", "LumB"]:
    mask = (subtype == st).values
    ax.scatter(meth_umap[mask, 0], meth_umap[mask, 1],
               c=COLORS[st], label=st, s=18, alpha=0.6, edgecolors="none")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("UMAP of PAM50 Promoter Methylation (post-imputation)")
ax.legend(title="Subtype")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../results/figures/meth_umap_by_subtype.png", dpi=300)
plt.show()

In [ ]:

# Mann-Whitney U test for ALL 50 PAM50 genes (not just the 8 shown in the violin
# panel above), with Benjamini-Hochberg correction since we're testing 50 genes
# at once. The 8-gene violin panel is illustrative, this table covers everything.

def bh_qvalues(pvals):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(q, 0, 1)
    return out

rows = []
for gene in rna.columns:
    luma = rna.loc[subtype == "LumA", gene].dropna()
    lumb = rna.loc[subtype == "LumB", gene].dropna()
    stat, pval = stats.mannwhitneyu(luma, lumb, alternative="two-sided")
    rows.append({"gene": gene, "p_value": pval, "median_LumA": luma.median(), "median_LumB": lumb.median()})

rna_sig = pd.DataFrame(rows)
rna_sig["q_value"] = bh_qvalues(rna_sig["p_value"])
rna_sig = rna_sig.sort_values("p_value")
rna_sig.to_csv("../results/tables/rna_subtype_significance_all_genes.csv", index=False)

print(f"Significant genes (q < 0.05): {(rna_sig['q_value'] < 0.05).sum()} / {len(rna_sig)}")
rna_sig.head(10)

In [ ]:

# Methylation violin plots, same 8 genes as the RNA violin panel, so the two are
# directly comparable. Uses mean promoter beta per gene (same as the scatter panel).

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for ax, gene in zip(axes, available):
    cpg_ids = cpg_gene.loc[cpg_gene["gene"] == gene, "cpg"].tolist()
    cpg_ids = [c for c in cpg_ids if c in meth_clean.columns]
    if not cpg_ids:
        ax.set_visible(False)
        continue

    mean_beta = meth_clean[cpg_ids].mean(axis=1)
    luma = mean_beta.loc[subtype.index[subtype == "LumA"]].dropna()
    lumb = mean_beta.loc[subtype.index[subtype == "LumB"]].dropna()

    parts = ax.violinplot([luma, lumb], positions=[0, 1], showmedians=True, showextrema=False)
    for pc, color in zip(parts["bodies"], [COLORS["LumA"], COLORS["LumB"]]):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    parts["cmedians"].set_color("black")
    parts["cmedians"].set_linewidth(1.5)

    rng = np.random.default_rng(42)
    for pos, vals, color in zip([0, 1], [luma, lumb], [COLORS["LumA"], COLORS["LumB"]]):
        idx = rng.choice(len(vals), size=min(80, len(vals)), replace=False)
        jitter = rng.uniform(-0.07, 0.07, size=len(idx))
        ax.scatter(pos + jitter, vals.iloc[idx], s=5, alpha=0.4, color=color)

    _, pval = stats.mannwhitneyu(luma, lumb, alternative="two-sided")
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"

    ax.set_title(f"{gene}  {sig}", fontsize=11)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["LumA", "LumB"])
    ax.set_ylabel("mean promoter beta")
    ax.spines[["top", "right"]].set_visible(False)

for ax in axes[len(available):]:
    ax.set_visible(False)

fig.suptitle("PAM50 Promoter Methylation by Luminal Subtype\n(Mann-Whitney U: * p<0.05, ** p<0.01, *** p<0.001)", fontsize=12)
plt.tight_layout()
plt.savefig("../results/figures/meth_violin_by_subtype.png", dpi=300)
plt.show()